# Nama: Andika Dinata
# NIM: 123140096

## Soal 1: Pasangan sintetis dan baseline InfoNCE

In [29]:
import numpy as np

SEED, N, DIM, TAU = 42, 6, 4, 0.10
rng = np.random.default_rng(SEED)

# shared latent factor, centered
z = rng.normal(0.0, 1.0, size=(N, DIM))
z = z - z.mean(axis=0, keepdims=True)

# the second modality sees the same factor through an orthogonal rotation Q
Q, _ = np.linalg.qr(rng.normal(0.0, 1.0, size=(DIM, DIM)))

image = z + 0.10 * rng.normal(size=(N, DIM))
text = z @ Q.T + 0.10 * rng.normal(size=(N, DIM))


def normalize(X):
    return X / np.linalg.norm(X, axis=1, keepdims=True)


def similarity_matrix(A, B):
    return normalize(A) @ normalize(B).T


def infonce(S, tau=TAU):
    Z = S / tau
    Z = Z - Z.max(axis=1, keepdims=True)  # numerical stability
    log_softmax = Z - np.log(np.exp(Z).sum(axis=1, keepdims=True))
    return -np.diag(log_softmax).mean()


def recall_at_1(S):
    return int(np.sum(np.argmax(S, axis=1) == np.arange(len(S))))

In [30]:
S0 = similarity_matrix(image, text)
print("diagonal            :", np.round(np.diag(S0), 4))
print("mean off-diagonal   : %+.3f" % S0[~np.eye(N, dtype=bool)].mean())
print("InfoNCE             : %.4f" % infonce(S0))
print("Recall@1            : %d/%d" % (recall_at_1(S0), N))

diagonal            : [-0.6085 -0.7713 -0.1997  0.5463 -0.6575 -0.6115]
mean off-diagonal   : +0.027
InfoNCE             : 10.6138
Recall@1            : 1/6


**Jawaban.** Kedua modalitas memang membawa faktor laten `z` yang sama, tapi teks melihatnya lewat rotasi `Q`, jadi koordinat di ruang teks tidak punya arti yang sama dengan koordinat di ruang gambar. Cosine similarity membandingkan dimensi ke-i dengan dimensi ke-i, sehingga yang dibandingkan adalah sumbu yang tidak berkorespondensi. Hasilnya nilai diagonal tidak lebih tinggi dari nilai lain, bahkan beberapa negatif. InfoNCE sekitar 10,6 itu jauh di atas ln(6) = 1,79 (tebakan acak), dan Recall@1 hanya 1/6 yang setara dengan chance level.

## Soal 2: Penyelarasan bentuk tertutup (Procrustes)

In [31]:
M = text.T @ image
U, s, Vt = np.linalg.svd(M)
R = U @ Vt
text_aligned = text @ R

S1 = similarity_matrix(image, text_aligned)
print("diagonal            :", np.round(np.diag(S1), 4))
print("mean off-diagonal   : %+.3f" % S1[~np.eye(N, dtype=bool)].mean())
print("InfoNCE             : %.4f" % infonce(S1))
print("Recall@1            : %d/%d" % (recall_at_1(S1), N))

diagonal            : [0.9972 0.9995 0.9985 0.9998 0.9611 0.8648]
mean off-diagonal   : -0.114
InfoNCE             : 0.1618
Recall@1            : 6/6


In [32]:
off = ~np.eye(N, dtype=bool)
rows = [
    ("Mean diagonal",     "%+.4f" % np.diag(S0).mean(), "%+.4f" % np.diag(S1).mean()),
    ("Mean off-diagonal", "%+.4f" % S0[off].mean(),     "%+.4f" % S1[off].mean()),
    ("InfoNCE (tau=0.10)", "%.4f" % infonce(S0),        "%.4f" % infonce(S1)),
    ("Recall@1",          "%d/%d" % (recall_at_1(S0), N), "%d/%d" % (recall_at_1(S1), N)),
]
print("%-20s %-12s %-12s" % ("Metric", "Before", "After"))
print("-" * 46)
for name, before, after in rows:
    print("%-20s %-12s %-12s" % (name, before, after))

Metric               Before       After       
----------------------------------------------
Mean diagonal        -0.3837      +0.9702     
Mean off-diagonal    +0.0268      -0.1143     
InfoNCE (tau=0.10)   10.6138      0.1618      
Recall@1             1/6          6/6         


**Jawaban.** Yang berubah hanya orientasi ruang teks: `R` memutar vektor teks sehingga sumbunya sejajar dengan ruang gambar. Informasinya tidak bertambah atau berkurang, karena rotasi ortogonal menjaga norma dan sudut antar vektor teks. Karena hubungan sebenarnya memang rotasi, solusi Procrustes lewat SVD memberi `R` yang optimal dalam bentuk tertutup. Setelah itu similarity pasangan benar naik mendekati 1 dan pasangan salah turun, dan untuk retrieval cukup dengan pasangan benar punya skor tertinggi di tiap baris, jadi Recall@1 jadi 6/6 dan InfoNCE turun ke sekitar 0,16. Satu catatan, `R` di sini dihitung dari 6 pasangan yang sama dengan yang dievaluasi, jadi hasilnya optimistis dibanding data baru.

## Soal 3: Pengaruh suhu

In [33]:
print("%-8s %-10s" % ("tau", "InfoNCE"))
print("-" * 20)
for t in (0.05, 0.10, 0.50):
    print("%-8.2f %-10.4f" % (t, infonce(S1, tau=t)))

tau      InfoNCE   
--------------------
0.05     0.1013    
0.10     0.1618    
0.50     0.6265    


**Jawaban.** Suhu kecil membuat logits `S / tau` jadi besar, sehingga softmax makin tajam dan probabilitas pasangan benar mendekati 1, jadi loss turun. Tapi turunnya loss ini sebagian hanya efek skala, karena similarity-nya sendiri tidak berubah. Pada pelatihan, tau yang terlalu kecil membuat gradient didominasi beberapa hard negative atau pasangan yang noisy, dan kesalahan sekecil apa pun dihukum sangat besar sehingga training jadi tidak stabil dan rawan overfit. Sebaliknya tau yang besar (0,50) membuat distribusi terlalu datar dan loss tetap tinggi, jadi perlu nilai di tengah.

## Soal 4: Retrieval dan Recall@K

In [34]:
# each row of S1.T is one text query over all images
S_text_to_image = S1.T
for i in range(N):
    rank = np.argsort(-S_text_to_image[i])
    pos = int(np.where(rank == i)[0][0]) + 1
    print("text %d: rank %d" % (i, pos))

print("Recall@1: %d/%d" % (recall_at_1(S_text_to_image), N))

text 0: rank 1
text 1: rank 1
text 2: rank 1
text 3: rank 1
text 4: rank 1
text 5: rank 1
Recall@1: 6/6


**Jawaban.** Semua kueri teks menempatkan gambar yang benar di peringkat 1, jadi Recall@1 = 6/6. Pada katalog ratusan ribu foto, banyak produk yang sama-sama cocok dengan satu deskripsi, sementara hanya satu yang dianggap benar. Gambar yang relevan sering jatuh di peringkat 2 sampai 5, sehingga Recall@1 terlihat rendah padahal modelnya baik. Karena itu Recall@5 atau Recall@10 biasanya lebih representatif.

Salah satu cara menaikkan Recall@K tanpa melatih ulang adalah query expansion: beberapa paraphrase dari deskripsi pelanggan di-embed lalu dirata-rata, sehingga vektor kueri lebih stabil terhadap pilihan kata.

## Soal 5: Analisis kritis

### (a) Perbandingan pola arsitektur

- **Dual-stream (mis. ViLBERT):** modalitas bertemu di dalam dua jalur paralel lewat co-attention. Interaksinya dalam dan berulang, dengan pertukaran dua arah tiap beberapa lapisan. Kurang cocok untuk large-scale retrieval karena tiap pasangan harus diproses penuh.
- **Multi-encoder (mis. LXMERT):** modalitas bertemu di satu cross-modality encoder setelah dua encoder unimodal. Interaksinya terpusat di satu blok cross-attention dua arah. Kurang cocok untuk large-scale retrieval, alasannya sama.
- **Dual-encoder (mis. CLIP):** modalitas tidak bertemu di dalam model, hanya di ruang embedding, sehingga tidak ada interaksi antar lapisan. Paling cocok untuk large-scale retrieval karena embedding bisa dihitung dan diindeks lebih dulu.

Untuk masalah di Bagian 4, saya memilih dual-encoder. Katalog berisi ratusan ribu foto dengan latensi di bawah satu detik, jadi embedding gambar harus dihitung dan diindeks sebelumnya, lalu kueri cukup di-encode sekali dan dicari lewat nearest neighbor search. Dual-stream dan multi-encoder memproses tiap pasangan kueri-gambar bersama, sehingga ratusan ribu forward pass per kueri tidak mungkin masuk satu detik. Tidak ada data berpasangan berlabel pun tidak masalah karena CLIP sudah pra-latih kontrastif pada 400 juta pasangan dan bisa dipakai zero-shot. Harganya adalah relasi dan negasi yang lemah, yang bisa dikurangi dengan re-ranking top-K memakai model yang interaksinya lebih dalam.

### (b) Uji kegagalan komposisional

In [35]:
# pip install torch transformers pillow
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor

MODEL_NAME = "openai/clip-vit-base-patch32"
IMAGE_PATH = "images.jpg"

model = CLIPModel.from_pretrained(MODEL_NAME)
processor = CLIPProcessor.from_pretrained(MODEL_NAME)
model.eval()

img = Image.open(IMAGE_PATH).convert("RGB")
candidates = [
    "sebuah sepeda di atas mobil",     # correct relation
    "sebuah mobil di atas sepeda",     # reversed relation
    "sebuah sepeda dan sebuah mobil",  # no relation
    "sebuah mobil tanpa sepeda",       # negation
]

inputs = processor(text=candidates, images=img, return_tensors="pt", padding=True)
with torch.no_grad():
    out = model(**inputs)

probs = out.logits_per_image.softmax(dim=1)[0]
order = torch.argsort(probs, descending=True)
lines = ["%-32s %.4f" % (candidates[int(i)], float(probs[int(i)])) for i in order]
print("\n".join(lines))

# probabilities saved for submission (mgg09_tugas_01.txt)
with open("mgg09_tugas_01.txt", "w") as f:
    f.write("\n".join(lines) + "\n")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

sebuah sepeda dan sebuah mobil   0.3164
sebuah mobil di atas sepeda      0.2905
sebuah sepeda di atas mobil      0.2130
sebuah mobil tanpa sepeda        0.1801


**Hasil run saya:** tanpa relasi 0,3164 (menang), relasi dibalik 0,2905, relasi benar 0,2130, negasi 0,1801 (terendah). Semuanya masih dekat 0,25, yaitu tebakan acak untuk empat kandidat.

**Jawaban.** Kegagalan ini paling masuk akal berasal dari tujuan pelatihannya, bukan dari kurangnya data atau kapasitas. Relasi dibalik (0,2905) dan tanpa relasi (0,3164) sama-sama mengungguli relasi yang benar (0,2130) padahal kata-katanya sama, jadi model hanya merespons kehadiran sepeda dan mobil, bukan susunannya. Negasi memang paling rendah, tapi jaraknya ke relasi yang benar kecil dan kalimatnya tetap memuat kedua objek, jadi saya tidak menyimpulkan model memahami negasi. Pada InfoNCE negative-nya kebanyakan scene lain, sehingga mengenali objek saja sudah cukup untuk menurunkan loss, dan hard negative seperti "sepeda di atas mobil" versus kebalikannya jarang muncul. Kurang data tidak cocok karena CLIP dilatih pada 400 juta pasangan, dan kapasitas juga tidak cocok karena encoder yang sama bagus untuk tugas lain seperti zero-shot classification. Dasar penilaian saya, sejalan dengan materi Pertemuan 9, adalah bahwa menyelaraskan seluruh gambar dengan seluruh kalimat lewat satu vektor global tidak menangkap relasi antar bagian.

(Catatan: CLIP ViT-B/32 dilatih dengan teks Inggris dan selisih probabilitasnya kecil dengan hanya satu gambar, jadi urutannya bersifat indikatif.)

### (c) Kaitan dengan teori

Pada pilar alignment (Pertemuan 7), cross-attention menghubungkan bagian yang berkorespondensi antar modalitas, misalnya query teks terhadap patch visual, sehingga relasi seperti "di atas" bisa dipelajari di level token. Dual-encoder tidak punya mekanisme ini karena modalitasnya hanya bertemu di ruang embedding, dan itu yang terlihat gagal pada Soal 5b. Penambahan modalitas juga tidak otomatis meningkatkan kinerja, seperti pada Soal 1 saat dua modalitas yang berisi informasi sama belum diselaraskan dan retrieval-nya tidak lebih baik dari tebakan acak.

Kasus pencarian produk menurut saya menguntungkan bagi dual-encoder, karena foto dan deskripsi membicarakan produk yang sama, kueri umumnya berisi objek dan atribut, dan skala serta latensi menuntut embedding yang dihitung di muka. Pilihan ini gagal kalau kueri bergantung pada relasi, negasi, atau penempelan atribut, misalnya "sepatu biru di sebelah kiri tas merah".